In [1]:
# CELL 1: data loader + text cleaning (Anadi, TF-IDF family)
import os, re, glob, random, unicodedata
import numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

# ---- find data (Kaggle first, then current folder) ----
hits = glob.glob("/kaggle/input/**/sample.csv", recursive=True) + glob.glob("sample.csv")
DATA_DIR = os.path.dirname(hits[0]) if hits else "."
train = pd.read_csv(os.path.join(DATA_DIR, "sample.csv"))
test_path = os.path.join(DATA_DIR, "test.csv")
test = pd.read_csv(test_path) if os.path.exists(test_path) else None
print("DATA_DIR:", DATA_DIR, "| train:", train.shape, "| test:", None if test is None else test.shape)

# ---- cleaning ----
# Unicode lookalikes (Cyrillic/Greek -> Latin) seen in the data, e.g. "sаlаry", "Amі"
LOOK = str.maketrans({
    "а":"a","е":"e","о":"o","р":"p","с":"c","у":"y","х":"x","і":"i","ј":"j","ѕ":"s","һ":"h","ԁ":"d","ԛ":"q","ѵ":"v",
    "А":"a","Е":"e","О":"o","Р":"p","С":"c","Х":"x","І":"i","Т":"t","В":"b","М":"m","Н":"h","К":"k",
    "α":"a","ο":"o","ρ":"p","ν":"v","ι":"i","τ":"t","κ":"k"})
ZERO_WIDTH = re.compile("[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad]")
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w-]+(?:\[\.\]|\.)(?:com|net|org|top|xyz|info|link|click|bd|me|co)\b\S*", re.I)
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF]")
BANGLISH_WORDS = set("apnar apni ami tumi korun kore koren hobe hoye hoyeche taka tk pathan pathaisi ferot din dibo "
    "ache ase nai na ki kothay bhai mama sob ekta eta ekhon ekhoni jitechen jitechne peyechen bolen bolben dao dey "
    "diye jan click link bhul vul bipode ghontar moddhe jonno theke dial e er ta te".split())
VOWELS = set("aeiouy")

def clean_one(msg: str) -> str:
    s = unicodedata.normalize("NFKC", str(msg))
    s = ZERO_WIDTH.sub("", s)
    s = unicodedata.normalize("NFC", s)               # Bangla: consistent ড়/ঢ়/য় form
    s = s.translate(LOOK).translate(BN_DIGITS).lower()
    s = s.replace("hxxp", "http")
    s = URL_RE.sub(lambda m: " urltok tld_" + re.sub(r".*\.([a-z]+)\W*$", r"\1", m.group(0).replace("[.]", ".")) + " ", s)
    s = EMOJI_RE.sub(" emojitok ", s)
    s = re.sub(r"(.)\1{2,}", r"\1", s)                # lateee -> late, PIIIN -> pin
    s = re.sub(r"\d[\d,\.]*", lambda m: f" num{len(re.sub(r'[^0-9]', '', m.group(0)))} ", s)
    s = re.sub(r"[^\w\u0980-\u09FF\s\*#]", " ", s)    # keep * and # (USSD codes like *247#)
    toks = []
    for t in s.split():                                # drop keyboard-mash noise strings
        if re.fullmatch(r"[a-z]{5,}", t) and re.search(r"[^aeiouy]{5,}", t):
            continue
        toks.append(t)
    return " ".join(toks)

def clean_text(text: str) -> str:
    """Keep conversation structure: clean each message, join with ' ||| ' ."""
    msgs = [clean_one(m) for m in str(text).split("|||")]
    out = " ||| ".join(m for m in msgs if m)
    return out if out.strip() else "emptytext"          # noise-only rows

def detect_lang(text: str) -> str:
    t = str(text)
    n_bn = len(re.findall(r"[\u0980-\u09FF]", t)); n_lat = len(re.findall(r"[A-Za-z]", t))
    if n_bn > 0 and n_lat < 0.3 * n_bn: return "bn"
    if n_bn > 0: return "mixed"
    words = re.findall(r"[a-z]+", t.lower())
    share = sum(w in BANGLISH_WORDS for w in words) / max(len(words), 1)
    return "banglish" if share >= 0.12 else "en"

def prep(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text_raw"] = df["text"].astype(str)
    df["text_clean"] = df["text_raw"].map(clean_text)
    df["lang"] = df["text_raw"].map(detect_lang)
    df["n_msgs"] = df["text_raw"].str.count(r"\|\|\|") + 1
    return df

train = prep(train); TECH = [f"T{i}" for i in range(1, 7)]
if test is not None: test = prep(test)
train["y"] = train["label"].astype(int)
train["act"] = train["action"].astype(int)
for t in TECH:
    train[t] = train["techniques"].map(lambda s: int(t in str(s).split("|")))

# ---- quick checks ----
print(train["y"].value_counts().sort_index().to_dict(), "| langs:", train["lang"].value_counts().to_dict(),
      "| conversations:", int((train["n_msgs"] > 1).sum()))
for i in [21, 26, 34, 52, 57]:
    print(f"[{train.loc[i,'y']}] {train.loc[i,'text_raw'][:70]!r}\n    -> {train.loc[i,'text_clean'][:90]!r}")

DATA_DIR: /kaggle/input/competitions/nsucec-neurahack-2026-preliminary | train: (86, 5) | test: (500000, 2)
{0: 32, 1: 12, 2: 10, 3: 11, 4: 9, 5: 12} | langs: {'en': 51, 'banglish': 29, 'bn': 6} | conversations: 8
[3] 'S: Hi, I saw your CV on Bdjobs. We wkrjfejejrn have a work-from-home p'
    -> 's hi i saw your cv on bdjobs we have a work from home position ||| r what iste salary ||| '
[3] 'Facccebook pge like korle protek 🔥 like e 5 tаkа. Gr\u200boup e jоin korun.'
    -> 'facebook pge like korle protek emojitok like e num1 taka group e join korun'
[1] 'Amі agnt, bhulnumber-e taka pathaisi. Ferot dіn,PIN ta bolen.'
    -> 'ami agnt bhulnumber e taka pathaisi ferot din pin ta bolen'
[1] 'mnbvcx Your bKash PIIIN will expire today Rpely witttth your currentP1'
    -> 'your bkash pin will expire today rpely with your currentp num1 n ad a new pin to emojitok '
[0] 'lkjsdfoiu xcvbnm'
    -> 'emptytext'


In [2]:
# CELL 2: TF-IDF baseline for all tasks (A label, B techniques, C action, D confidence) + grouped CV + submission
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from scipy.sparse import hstack
from scipy.sparse.csgraph import connected_components

assert test is not None, "test.csv not found: attach the competition data with Add Input and rerun Cell 1"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

# near-duplicate groups (char TF-IDF cosine > 0.8) so CV does not leak templates
_gv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
_G = _gv.fit_transform(train["text_clean"])
_S = (_G @ _G.T).tocsr(); _S.data = (_S.data > 0.8).astype(int); _S.eliminate_zeros()
_, groups = connected_components(_S, directed=False)

def _proba(model, X, n_classes):
    p = np.zeros((X.shape[0], n_classes)); p[:, model.classes_] = model.predict_proba(X); return p

def fit_predict(tr, te_texts):
    """Fit all heads on train rows `tr` (DataFrame), return probabilities for te_texts. Vectorizers fit on train rows only."""
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True)
    wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"\S+")
    Xtr = hstack([cv.fit_transform(tr["text_clean"]), wv.fit_transform(tr["text_clean"])]).tocsr()
    Xte = hstack([cv.transform(te_texts), wv.transform(te_texts)]).tocsr()
    lr = lambda: LogisticRegression(C=20, class_weight="balanced", max_iter=3000, random_state=SEED)
    out = {"label": _proba(lr().fit(Xtr, tr["y"]), Xte, 6), "action": _proba(lr().fit(Xtr, tr["act"]), Xte, 6)}
    tech = np.zeros((Xte.shape[0], 6))
    for j, t in enumerate(TECH):
        if tr[t].nunique() == 2:
            tech[:, j] = lr().fit(Xtr, tr[t]).predict_proba(Xte)[:, 1]
        else:
            tech[:, j] = float(tr[t].iloc[0])
    out["tech"] = tech
    return out

# ---- grouped 5-fold CV (seed 42), real rows only ----
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = {"label": np.zeros((len(train), 6)), "action": np.zeros((len(train), 6)), "tech": np.zeros((len(train), 6))}
fold_id = np.zeros(len(train), dtype=int)
for k, (a, b) in enumerate(skf.split(train, train["y"], groups)):
    r = fit_predict(train.iloc[a], train["text_clean"].iloc[b])
    for key in oof: oof[key][b] = r[key]
    fold_id[b] = k

def decode(label_p, action_p, tech_p):
    lab = label_p.argmax(1)
    act = np.where(lab == 0, 0, action_p[:, 1:].argmax(1) + 1)   # legit -> 0, scam -> best of 1..5
    techs = []
    for i in range(len(lab)):
        if lab[i] == 0: techs.append("NONE"); continue
        s = [TECH[j] for j in range(6) if tech_p[i, j] >= 0.5] or [TECH[int(tech_p[i].argmax())]]
        techs.append("|".join(s))
    return lab, act, techs

lab_o, act_o, tech_o = decode(oof["label"], oof["action"], oof["tech"])
Ytech = train[TECH].values
Ptech = np.array([[int(t in s.split("|")) for t in TECH] for s in tech_o])
print("CV macro-F1  A(label): %.3f | B(tech): %.3f | C(action): %.3f" % (
    f1_score(train["y"], lab_o, average="macro"), f1_score(Ytech, Ptech, average="macro", zero_division=0),
    f1_score(train["act"], act_o, average="macro")))
print("per-class F1 (label 0..5):", np.round(f1_score(train["y"], lab_o, average=None, labels=range(6)), 2))
for lg, d in train.groupby("lang"):
    print(f"  lang {lg:9s} n={len(d):3d} macro-F1={f1_score(d['y'], lab_o[d.index], average='macro', labels=range(6), zero_division=0):.3f}")

# ---- Task D: calibrate confidence from out-of-fold correctness ----
mp_o = oof["label"].max(1, keepdims=True)
calib = LogisticRegression(C=10, random_state=SEED).fit(mp_o, (lab_o == train["y"]).astype(int))
print("OOF accuracy: %.3f | mean calibrated confidence: %.3f" % ((lab_o == train["y"]).mean(), calib.predict_proba(mp_o)[:, 1].mean()))

# ---- final fit on all train rows -> test ----
rt = fit_predict(train, test["text_clean"])
lab_t, act_t, tech_t = decode(rt["label"], rt["action"], rt["tech"])
conf_t = np.clip(calib.predict_proba(rt["label"].max(1, keepdims=True))[:, 1], 0.05, 0.98)
sub = pd.DataFrame({"id": test["id"], "label": lab_t.astype(int), "techniques": tech_t,
                    "action": act_t.astype(int), "confidence": np.round(conf_t, 4)})
sub.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
np.save(os.path.join(OUT_DIR, "oof_tfidf.npy"), oof["label"]); np.save(os.path.join(OUT_DIR, "test_tfidf.npy"), rt["label"])
print("saved submission.csv", sub.shape, "| test label mix:", np.round(np.bincount(lab_t, minlength=6) / len(lab_t), 3))

CV macro-F1  A(label): 0.565 | B(tech): 0.434 | C(action): 0.411
per-class F1 (label 0..5): [0.75 0.69 0.5  0.44 0.62 0.38]
  lang banglish  n= 29 macro-F1=0.584
  lang bn        n=  6 macro-F1=0.000
  lang en        n= 51 macro-F1=0.523
OOF accuracy: 0.628 | mean calibrated confidence: 0.628
saved submission.csv (500000, 5) | test label mix: [0.466 0.101 0.081 0.108 0.108 0.137]


In [3]:
# CELL 3: validate submission.csv (5-column ScamShield BD format)
sub = pd.read_csv(os.path.join(OUT_DIR, "submission.csv"), keep_default_na=False)   # keep "NONE" as text
tst = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
assert list(sub.columns) == ["id", "label", "techniques", "action", "confidence"], list(sub.columns)
assert len(sub) == len(tst) and sub["id"].is_unique and set(sub["id"]) == set(tst["id"])
assert pd.api.types.is_integer_dtype(sub["label"]) and sub["label"].between(0, 5).all()
assert pd.api.types.is_integer_dtype(sub["action"]) and sub["action"].between(0, 5).all()
assert pd.to_numeric(sub["confidence"]).between(0, 1).all()
_valid = {"T1", "T2", "T3", "T4", "T5", "T6"}
assert sub["techniques"].map(lambda s: s == "NONE" or (s != "" and set(s.split("|")) <= _valid)).all(), "bad techniques"
print("OK: submission.csv passes all format checks"); print(sub["label"].value_counts(normalize=True).sort_index().round(3))

OK: submission.csv passes all format checks
label
0    0.466
1    0.101
2    0.081
3    0.108
4    0.108
5    0.137
Name: proportion, dtype: float64


In [4]:
import os
for root, _, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/sample_submission.csv
/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/sample.csv
/kaggle/input/competitions/nsucec-neurahack-2026-preliminary/test.csv


In [5]:
import glob
D = os.path.dirname(glob.glob("/kaggle/input/**/sample.csv", recursive=True)[0])
print("D =", D, os.listdir(D))

D = /kaggle/input/competitions/nsucec-neurahack-2026-preliminary ['sample_submission.csv', 'sample.csv', 'test.csv']


In [6]:
import os, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

SEED = 42
np.random.seed(SEED)
D = "/kaggle/input/competitions/nsucec-neurahack-2026-preliminary"
tr = pd.read_csv(f"{D}/sample.csv")
te = pd.read_csv(f"{D}/test.csv")
ss = pd.read_csv(f"{D}/sample_submission.csv")
print("test cols:", list(te.columns), "| rows:", len(te))
assert "text" in te.columns and len(te) == len(ss) and set(te.id) == set(ss.id)

cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), lowercase=True, sublinear_tf=True, dtype=np.float32)
wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=True, sublinear_tf=True, dtype=np.float32)
Xtr = hstack([cv.fit_transform(tr.text), wv.fit_transform(tr.text)]).tocsr()

def fit(y):
    return LogisticRegression(C=10, max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, y)
m_label, m_action = fit(tr.label), fit(tr.action)
tech_map = tr.groupby("label").techniques.agg(lambda s: s.value_counts().index[0]).to_dict()

labs, acts, confs = [], [], []
for i in range(0, len(te), 50000):
    t = te.text.iloc[i:i+50000].fillna("").astype(str)
    X = hstack([cv.transform(t), wv.transform(t)]).tocsr()
    p = m_label.predict_proba(X)
    labs.append(m_label.classes_[p.argmax(1)]); confs.append(p.max(1))
    acts.append(m_action.predict(X))
lab = np.concatenate(labs); act = np.concatenate(acts); conf = np.concatenate(confs)
act = np.where(np.isin(lab, [0, 5]), 0, act)
tech = [tech_map[l] if l not in (0, 5) else "NONE" for l in lab]

sub = pd.DataFrame({"id": te.id, "label": lab.astype(int), "techniques": tech,
                    "action": act.astype(int), "confidence": conf.round(4)})
sub = sub.set_index("id").loc[ss.id].reset_index()   # same id order as sample_submission
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.shape, list(sub.columns))
print(sub.label.value_counts(normalize=True).sort_index().round(3))

test cols: ['id', 'text'] | rows: 500000
(500000, 5) ['id', 'label', 'techniques', 'action', 'confidence']
label
0    0.502
1    0.117
2    0.074
3    0.076
4    0.119
5    0.112
Name: proportion, dtype: float64


In [7]:
!git clone https://github.com/fairuz-anadi/neurahack.git /kaggle/working/repo
!ls -R /kaggle/working/repo | head -50
!cd /kaggle/working/repo && git log --oneline | head -5

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 115 (delta 34), reused 113 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 100.41 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/kaggle/working/repo:
artifacts
augmentation
common
configs
data
docs
folds
models
notebooks
outputs
README.md
requirements.txt
scripts
tests

/kaggle/working/repo/artifacts:

/kaggle/working/repo/augmentation:
corrupt.py
__init__.py

/kaggle/working/repo/common:
data_loader.py
folds.py
__init__.py
labels.py
metrics.py
prediction_io.py
preprocessing.py
schema.py
seed.py
stress.py
submission.py

/kaggle/working/repo/configs:
base.yaml
labels.yaml
model_configs

/kaggle/working/repo/configs/model_configs:
tfidf_lr.yaml

/kaggle/working/repo/data:

/kaggle/working/repo/docs:
EXPERIMENT_LOG.md
RESOURCE_DECLARATION.md
TEAM_CONTRACT.md

a93cd

In [8]:
!cd /kaggle/working/repo && git pull

Already up to date.


In [9]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{tok}@github.com/fairuz-anadi/neurahack.git /kaggle/working/repo
!mkdir -p /kaggle/working/repo/anadi
!cp /kaggle/working/submission.csv /kaggle/working/oof_tfidf.npy /kaggle/working/test_tfidf.npy /kaggle/working/repo/anadi/
!cd /kaggle/working/repo && git config user.name "Anadi" && git config user.email "you@example.com" \
 && git checkout -b anadi-tfidf && git add . && git commit -m "add tfidf baseline" && git push origin anadi-tfidf


fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.
Switched to a new branch 'anadi-tfidf'
On branch anadi-tfidf
nothing to commit, working tree clean


In [10]:
import numpy as np, pandas as pd
from scipy.sparse import hstack
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
D = "/kaggle/input/competitions/nsucec-neurahack-2026-preliminary"
tr = pd.read_csv(f"{D}/sample.csv")
y = tr.label.values

# groups: near-duplicates (char tfidf cosine > 0.8) share a group
g_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True)
G = g_vec.fit_transform(tr.text)
sim = (G @ G.T).toarray()
_, groups = connected_components(sim > 0.8, directed=False)
print("rows:", len(tr), "| groups:", len(set(groups)))

def feats(train_txt, other_txt):
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True, dtype=np.float32)
    wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
    a = hstack([cv.fit_transform(train_txt), wv.fit_transform(train_txt)]).tocsr()
    b = hstack([cv.transform(other_txt), wv.transform(other_txt)]).tocsr()
    return a, b

scores, oof0 = [], None
for rep in range(5):
    seed = SEED + rep
    oof = np.zeros((len(tr), 6))
    for trn, val in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(tr.text, y, groups):
        A, B = feats(tr.text.iloc[trn], tr.text.iloc[val])
        m = LogisticRegression(C=10, max_iter=2000, class_weight="balanced", random_state=SEED).fit(A, y[trn])
        p = np.zeros((len(val), 6)); p[:, m.classes_] = m.predict_proba(B)
        oof[val] = p
    scores.append(f1_score(y, oof.argmax(1), average="macro"))
    if rep == 0: oof0 = oof
print("macro-F1 per repeat:", np.round(scores, 3))
print(f"CV macro-F1 = {np.mean(scores):.3f} +/- {np.std(scores):.3f}  (LR char+word, 5-fold x5, grouped)")
np.save("/kaggle/working/oof_tfidf.npy", oof0)
np.save("/kaggle/working/groups.npy", groups)

rows: 86 | groups: 86
macro-F1 per repeat: [0.428 0.493 0.452 0.52  0.547]
CV macro-F1 = 0.488 +/- 0.043  (LR char+word, 5-fold x5, grouped)


In [11]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
!git clone https://{tok}@github.com/fairuz-anadi/neurahack.git /kaggle/working/repo
!mkdir -p /kaggle/working/repo/anadi
!cp /kaggle/working/submission.csv /kaggle/working/oof_tfidf.npy /kaggle/working/test_tfidf.npy /kaggle/working/repo/anadi/
!cd /kaggle/working/repo && git config user.name "Anadi" && git config user.email "you@example.com" \
 && git checkout -b anadi-tfidf && git add . && git commit -m "add tfidf baseline" && git push origin anadi-tfidf


fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.
fatal: a branch named 'anadi-tfidf' already exists


In [12]:
import re, numpy as np, pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
D = "/kaggle/input/competitions/nsucec-neurahack-2026-preliminary"
tr = pd.read_csv(f"{D}/sample.csv"); y = tr.label.values
oof = np.load("/kaggle/working/oof_tfidf.npy"); groups = np.load("/kaggle/working/groups.npy")
pred = oof.argmax(1)

# language heuristic (no language column): Bangla script -> bn, Banglish keywords -> banglish, else en
BL = set("apnar apni apnake korun kore korte tk taka sob hobe jonno ache nai nei amar tomar ekhane ekhon jan pathan pathiye bhai dhonnobad dhonnobad jitechen jitesen pabe pan nin".split())
def lang(t):
    if re.search(r"[\u0980-\u09FF]", t): return "bn"
    return "banglish" if len(set(re.findall(r"[a-z]+", t.lower())) & BL) >= 1 else "en"
tr["lang"] = tr.text.map(lang)
print(tr.lang.value_counts().to_dict(), "(heuristic)")

print("\nconfusion matrix (rows=true, cols=pred):\n", confusion_matrix(y, pred, labels=range(6)))
print("\n", classification_report(y, pred, labels=range(6), zero_division=0, digits=3))
for l, g in tr.groupby("lang"):
    print(f"lang {l:9s} n={len(g):3d} macro-F1={f1_score(y[g.index], pred[g.index], average='macro', labels=sorted(set(y[g.index])), zero_division=0):.3f}  acc={(y[g.index]==pred[g.index]).mean():.3f}")

tr["pred"] = pred; tr["p_true"] = oof[np.arange(len(tr)), y]
print("\nworst errors (lowest prob on the true class):")
print(tr.sort_values("p_true").head(8)[["label", "pred", "p_true", "text"]].to_string(max_colwidth=90))

# typo-noised validation copy (noise only on validation text, train stays clean)
rng = np.random.RandomState(SEED)
def noise(t, p=0.15):
    out = []
    for w in t.split():
        if len(w) > 3 and rng.rand() < p:
            i = rng.randint(1, len(w) - 1); k = rng.randint(3)
            w = w[:i] + w[i+1:] if k == 0 else (w[:i] + w[i] + w[i:] if k == 1 else w[:i] + w[i+1] + w[i] + w[i+2:])
        out.append(w)
    return " ".join(out)

clean_s, noisy_s = [], []
for rep in range(5):
    pc, pn = np.zeros(len(tr), int), np.zeros(len(tr), int)
    for trn, val in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + rep).split(tr.text, y, groups):
        cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True, dtype=np.float32)
        wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
        A = hstack([cv.fit_transform(tr.text.iloc[trn]), wv.fit_transform(tr.text.iloc[trn])]).tocsr()
        m = LogisticRegression(C=10, max_iter=2000, class_weight="balanced", random_state=SEED).fit(A, y[trn])
        for txts, store in ((tr.text.iloc[val], pc), (tr.text.iloc[val].map(noise), pn)):
            B = hstack([cv.transform(txts), wv.transform(txts)]).tocsr()
            store[val] = m.classes_[m.predict_proba(B).argmax(1)]
    clean_s.append(f1_score(y, pc, average="macro")); noisy_s.append(f1_score(y, pn, average="macro"))
print(f"\nclean CV macro-F1 = {np.mean(clean_s):.3f} | typo-noised = {np.mean(noisy_s):.3f} (drop {np.mean(clean_s)-np.mean(noisy_s):.3f})")

{'banglish': 47, 'en': 33, 'bn': 6} (heuristic)

confusion matrix (rows=true, cols=pred):
 [[27  1  1  1  1  1]
 [ 3  8  0  0  1  0]
 [ 2  1  5  0  1  1]
 [ 8  0  0  1  0  2]
 [ 4  0  2  0  3  0]
 [ 9  0  0  2  0  1]]

               precision    recall  f1-score   support

           0      0.509     0.844     0.635        32
           1      0.800     0.667     0.727        12
           2      0.625     0.500     0.556        10
           3      0.250     0.091     0.133        11
           4      0.500     0.333     0.400         9
           5      0.200     0.083     0.118        12

    accuracy                          0.523        86
   macro avg      0.481     0.420     0.428        86
weighted avg      0.486     0.523     0.478        86

lang banglish  n= 47 macro-F1=0.492  acc=0.553
lang bn        n=  6 macro-F1=0.000  acc=0.000
lang en        n= 33 macro-F1=0.296  acc=0.576

worst errors (lowest prob on the true class):
    label  pred    p_true                        

In [13]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
R = "/kaggle/working/repo"
!rm -rf {R}
!git clone https://{tok}@github.com/fairuz-anadi/neurahack.git {R}
!mkdir -p {R}/anadi
!cp /kaggle/working/oof_tfidf.npy /kaggle/working/groups.npy {R}/anadi/
!printf "*.csv\ntest_*.npy\n.env\n" > {R}/anadi/.gitignore
!cd {R} && git config user.name "Anadi" && git config user.email "you@example.com" \
 && git checkout -b anadi-tfidf && git add anadi && git commit -m "tfidf baseline: CV 0.488, oof + groups" && git push origin anadi-tfidf
!rm -rf {R}

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 115 (delta 34), reused 113 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 100.41 KiB | 1.86 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Switched to a new branch 'anadi-tfidf'
[anadi-tfidf 7d3dc2e] tfidf baseline: CV 0.488, oof + groups
 1 file changed, 3 insertions(+)
 create mode 100644 anadi/.gitignore
remote: Permission to fairuz-anadi/neurahack.git denied to fairuz-anadi.
fatal: unable to access 'https://github.com/fairuz-anadi/neurahack.git/': The requested URL returned error: 403


In [14]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
R = "/kaggle/working/repo"
!rm -rf {R}
!git clone https://{tok}@github.com/fairuz-anadi/neurahack.git {R}
!mkdir -p {R}/anadi
!ls -la /kaggle/working/*.npy
!cp /kaggle/working/oof_tfidf.npy /kaggle/working/groups.npy {R}/anadi/
!printf "*.csv\ntest_*.npy\n.env\n" > {R}/anadi/.gitignore
!cd {R} && git config user.name "Anadi" && git config user.email "you@example.com" \
 && git checkout -b anadi-tfidf && git add -f anadi && git status --short \
 && git commit -m "tfidf baseline: CV 0.488, oof + groups" && git push origin anadi-tfidf
!rm -rf {R}

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 115 (delta 34), reused 113 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 100.41 KiB | 1.93 MiB/s, done.
Resolving deltas: 100% (34/34), done.
-rw-r--r-- 1 root root      472 Nov 26 16:39 /kaggle/working/groups.npy
-rw-r--r-- 1 root root     4256 Nov 26 16:39 /kaggle/working/oof_tfidf.npy
-rw-r--r-- 1 root root 24000128 Nov 26 16:36 /kaggle/working/test_tfidf.npy
Switched to a new branch 'anadi-tfidf'
A  anadi/.gitignore
A  anadi/groups.npy
A  anadi/oof_tfidf.npy
[anadi-tfidf 1869167] tfidf baseline: CV 0.488, oof + groups
 3 files changed, 3 insertions(+)
 create mode 100644 anadi/.gitignore
 create mode 100644 anadi/groups.npy
 create mode 100644 anadi/oof_tfidf.npy
remote: Permission to fairuz-anadi/neurahack.git denied to fairuz-anadi.
fatal: unable to access 'https://gith

In [15]:
from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("GH_TOKEN")
print(len(tok), tok[:4])   # classic starts with ghp_, fine-grained with github_pat_
!curl -sI -H "Authorization: token {tok}" https://api.github.com/user | grep -i -E "^(HTTP|x-oauth-scopes|x-accepted)"

93 gith




In [16]:
# CELL 4: Anadi classical ML experiments (E002-E009) + artifact export
# Run AFTER Cell 1 (needs: train, test, TECH, clean_text, URL_RE, ZERO_WIDTH, EMOJI_RE).
# Validation-only by default. Set RUN_FULL = True to fit on all train rows and predict the real test (Kaggle only).
import os, re, json, math, hashlib, time
import numpy as np, pandas as pd
from scipy.sparse import hstack, csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler

SEED = 42
RUN_FULL = False
REPEATS = 3                      # repeated grouped CV (seeds 42, 43, 44)
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
FOLDS = None                     # TODO for Rabib-compatible artifacts: set to the repo's shared fold manifest as a list of (train_idx, val_idx)

# ---------------- groups (near-duplicate templates stay in one fold) ----------------
_gv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
_G = _gv.fit_transform(train["text_clean"])
_S = (_G @ _G.T).tocsr(); _S.data = (_S.data > 0.8).astype(int); _S.eliminate_zeros()
_, groups = connected_components(_S, directed=False)
print("rows", len(train), "| groups", len(set(groups)))

# ---------------- controlled noise augmentation (E005) ----------------
LOOK_REV = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "y": "у", "x": "х", "i": "і"}
MASH = ["wkrjfejejrn", "zzqwpo", "mnbvcx", "lkjsdfoiu", "xcvbnm", "qwrtpsdf"]

def corrupt_msg(m, rng, p=0.12):
    out = []
    for ch in m:
        r = rng.rand()
        if ch.isalpha() and r < p * 0.25: continue                                   # deletion
        if ch.isalpha() and r < p * 0.50: out.append(ch * rng.randint(2, 4)); continue  # repetition
        if ch.lower() in LOOK_REV and r < p * 0.75: out.append(LOOK_REV[ch.lower()]); continue  # lookalike
        if ch == " " and r < p: continue                                              # drop a space
        if ch.isalpha() and r < p * 0.85: out.append("​" + ch); continue         # invisible char
        out.append(ch)
    s = "".join(out); words = s.split(" ")
    if rng.rand() < 0.2 and words:                                                    # keyboard-mash word
        words.insert(rng.randint(0, len(words) + 1), MASH[rng.randint(len(MASH))])
    return " ".join(words)

def corrupt_text(t, rng):
    return " ||| ".join(corrupt_msg(m.strip(), rng) for m in str(t).split("|||"))

def corrupted_df(df, seed):
    rng = np.random.RandomState(seed)
    d = df.copy()
    d["text_raw"] = d["text_raw"].map(lambda t: corrupt_text(t, rng))
    d["text_clean"] = d["text_raw"].map(clean_text)
    return d

# ---------------- structured features (E004) ----------------
def _entropy(s):
    if not s: return 0.0
    _, c = np.unique(list(s), return_counts=True); p = c / c.sum()
    return float(-(p * np.log2(p)).sum())

def struct_feats(df):
    raw = df["text_raw"].astype(str); cl = df["text_clean"].astype(str)
    n = raw.str.len().clip(lower=1)
    f = pd.DataFrame(index=df.index)
    f["n_msgs"] = df["n_msgs"].values
    f["loglen"] = np.log1p(n.values)
    f["digit"] = raw.map(lambda s: sum(c.isdigit() for c in s)).values / n.values
    f["upper"] = raw.map(lambda s: sum(c.isupper() for c in s)).values / n.values
    f["nonascii"] = raw.map(lambda s: sum(ord(c) > 127 for c in s)).values / n.values
    f["url"] = raw.map(lambda s: int(bool(URL_RE.search(s)))).values
    f["bn"] = raw.map(lambda s: int(bool(re.search(r"[ঀ-৿]", s)))).values
    f["latin"] = raw.map(lambda s: int(bool(re.search(r"[A-Za-z]", s)))).values
    f["rep"] = raw.map(lambda s: len(re.findall(r"(.)\1{2,}", s))).values
    f["zw"] = raw.map(lambda s: len(ZERO_WIDTH.findall(s))).values
    f["ent"] = raw.map(_entropy).values
    f["kept"] = np.array([len(c) / max(len(r), 1) for c, r in zip(cl, raw)])
    f["emoji"] = raw.map(lambda s: len(EMOJI_RE.findall(s))).values
    f["ussd"] = raw.map(lambda s: int("*" in s and "#" in s)).values
    f["money"] = raw.str.contains(r"(?:\btk\b|taka|bdt|৳|টাকা)", case=False, regex=True).astype(int).values
    f["last_len"] = raw.map(lambda s: np.log1p(len(str(s).split("|||")[-1]))).values
    return f.values.astype(float)

# ---------------- model family ----------------
def _make_head(kind, y):
    cnt = np.bincount(y); present = cnt[cnt > 0]
    if kind == "svc" and present.min() >= 3 and len(present) >= 2:
        return CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced", random_state=SEED), cv=3, method="sigmoid")
    if kind == "cnb":
        return ComplementNB(alpha=0.3)
    return LogisticRegression(C=20, class_weight="balanced", max_iter=3000, random_state=SEED)

def _proba(m, X, n=6):
    p = np.zeros((X.shape[0], n)); p[:, m.classes_] = m.predict_proba(X); return p

class ClassicModel:
    """cfg: heads (list of 'lr'/'svc'/'cnb'), aug (noisy copies per train row), struct (bool)."""
    def __init__(self, cfg): self.cfg = cfg
    def _X(self, df, fit=False):
        txt = df["text_clean"]
        if fit:
            self.cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 5), sublinear_tf=True, dtype=np.float32)
            self.wv = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"\S+", dtype=np.float32)
            tf = hstack([self.cv.fit_transform(txt), self.wv.fit_transform(txt)]).tocsr()
        else:
            tf = hstack([self.cv.transform(txt), self.wv.transform(txt)]).tocsr()
        if not self.cfg.get("struct"): return tf, tf
        S = struct_feats(df)
        if fit: self.sc = StandardScaler().fit(S)
        S = np.clip(self.sc.transform(S), -5, 5) * 0.3
        return tf, hstack([tf, csr_matrix(S)]).tocsr()
    def fit(self, tr, seed=SEED):
        parts = [tr] + [corrupted_df(tr, seed * 1000 + k) for k in range(self.cfg.get("aug", 0))]
        tr = pd.concat(parts, ignore_index=True)
        tf, full = self._X(tr, fit=True)
        self.m = {}
        for kind in self.cfg["heads"]:
            X = tf if kind == "cnb" else full
            h = {"label": _make_head(kind, tr["y"].values).fit(X, tr["y"].values),
                 "action": _make_head(kind, tr["act"].values).fit(X, tr["act"].values), "tech": []}
            for t in TECH:
                yt = tr[t].values
                if yt.min() == yt.max(): h["tech"].append(float(yt[0]))
                else: h["tech"].append(_make_head(kind if np.bincount(yt).min() >= 3 else "lr", yt).fit(X, yt))
            self.m[kind] = h
        return self
    def predict(self, df):
        tf, full = self._X(df)
        out = {"label": 0, "action": 0, "tech": 0}; k = len(self.m)
        for kind, h in self.m.items():
            X = tf if kind == "cnb" else full
            out["label"] = out["label"] + _proba(h["label"], X) / k
            out["action"] = out["action"] + _proba(h["action"], X) / k
            tp = np.zeros((X.shape[0], 6))
            for j, mt in enumerate(h["tech"]):
                tp[:, j] = mt if isinstance(mt, float) else mt.predict_proba(X)[:, list(mt.classes_).index(1)]
            out["tech"] = out["tech"] + tp / k
        return out

# ---------------- decoding + scoring ----------------
def decode(lp, ap, tp, constrain):
    lab = lp.argmax(1); act = ap.argmax(1); techs = []
    for i in range(len(lab)):
        if constrain and lab[i] in (0, 5): act[i] = 0
        if lab[i] == 0 or (constrain and lab[i] == 5): techs.append("NONE"); continue
        s = [TECH[j] for j in range(6) if tp[i, j] >= 0.5] or [TECH[int(tp[i].argmax())]]
        techs.append("|".join(s))
    return lab, act, techs

def f1s(df, o, constrain):
    lab, act, techs = decode(o["label"], o["action"], o["tech"], constrain)
    P = np.array([[int(t in s.split("|")) for t in TECH] for s in techs])
    return (f1_score(df["y"], lab, average="macro", labels=range(6), zero_division=0),
            f1_score(df[TECH].values, P, average="macro", zero_division=0),
            f1_score(df["act"], act, average="macro", labels=range(6), zero_division=0))

def calib_score(lp, y):
    mp = lp.max(1, keepdims=True); ok = (lp.argmax(1) == y).astype(int)
    p = cross_val_predict(LogisticRegression(C=10, random_state=SEED), mp, ok, cv=5, method="predict_proba")[:, 1]
    return 1 - float(np.mean((p - ok) ** 2))      # 1 - Brier (higher is better), a proxy only

def cv_splits(rep):
    if FOLDS is not None: return FOLDS
    return list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + rep).split(train, train["y"], groups))

CONFIGS = {
    "E0 lr baseline":          {"heads": ["lr"], "aug": 0, "struct": False},
    "E1 lr +noise x1":         {"heads": ["lr"], "aug": 1, "struct": False},
    "E2 lr +noise x2":         {"heads": ["lr"], "aug": 2, "struct": False},
    "E3 svc +noise x1":        {"heads": ["svc"], "aug": 1, "struct": False},
    "E4 lr +struct +noise x1": {"heads": ["lr"], "aug": 1, "struct": True},
    "E5 lr+svc +noise x1":     {"heads": ["lr", "svc"], "aug": 1, "struct": False},
    "E6 lr+svc+cnb +noise x1": {"heads": ["lr", "svc", "cnb"], "aug": 1, "struct": False},
}

def run_config(name, cfg):
    res = {k: [] for k in ["A", "B0", "B1", "C0", "C1", "A_noisy", "calib"]}; first = None
    for rep in range(REPEATS):
        oof = {k: np.zeros((len(train), 6)) for k in ["label", "action", "tech"]}
        noisy = np.zeros((len(train), 6))
        for a, b in cv_splits(rep):
            m = ClassicModel(cfg).fit(train.iloc[a], seed=SEED + rep)
            r = m.predict(train.iloc[b])
            for k in oof: oof[k][b] = r[k]
            noisy[b] = m.predict(corrupted_df(train.iloc[b], 777 + rep))["label"]   # noise only on validation copies
        a0, b0, c0 = f1s(train, oof, False); a1, b1, c1 = f1s(train, oof, True)
        res["A"].append(a0); res["B0"].append(b0); res["B1"].append(b1); res["C0"].append(c0); res["C1"].append(c1)
        res["A_noisy"].append(f1_score(train["y"], noisy.argmax(1), average="macro", labels=range(6), zero_division=0))
        res["calib"].append(calib_score(oof["label"], train["y"].values))
        if rep == 0: first = oof
    return {k: float(np.mean(v)) for k, v in res.items()}, first

results, oofs = {}, {}
t0 = time.time()
for name, cfg in CONFIGS.items():
    results[name], oofs[name] = run_config(name, cfg)
    print(f"{name:26s} done {time.time() - t0:5.0f}s")
tbl = pd.DataFrame(results).T.round(3)
tbl.columns = ["A_F1", "B_F1(free)", "B_F1(NONE for 0,5)", "C_F1(free)", "C_F1(0 for 0,5)", "A_F1_noisy", "calib(1-Brier)"]
print("\nMean over %d repeated grouped 5-fold runs (real rows only; proxies, not Kaggle scores):" % REPEATS)
print(tbl.to_string())

# ---------------- pick winner: best A, tie -> best noisy A ----------------
BEST = (tbl["A_F1"] + 0.5 * tbl["A_F1_noisy"]).idxmax()
print("\nBEST config:", BEST)

# ---------------- artifact export (names follow TEAM_CONTRACT: oof_A/B/C_<name>.npy) ----------------
def save_artifacts(name, tag, oof, tst=None, test_ids=None):
    for task, key in (("A", "label"), ("B", "tech"), ("C", "action")):
        np.save(f"{OUT_DIR}/oof_{task}_{tag}.npy", oof[key].astype(np.float32))
        if tst is not None: np.save(f"{OUT_DIR}/test_{task}_{tag}.npy", tst[key].astype(np.float32))
    np.save(f"{OUT_DIR}/oof_ids.npy", train["id"].values)
    if test_ids is not None: np.save(f"{OUT_DIR}/test_ids.npy", test_ids)
    meta = {"model": tag, "config": CONFIGS[name], "seed": SEED, "oof_rows": int(len(train)),
            "columns": {"A": "classes 0-5", "B": "T1-T6 independent probabilities", "C": "actions 0-5"},
            "train_fingerprint": hashlib.md5((",".join(map(str, train["id"])) + ",".join(map(str, train["y"]))).encode()).hexdigest(),
            "folds": "shared manifest" if FOLDS is not None else "LOCAL groups (not the shared folds): validate before giving to Rabib"}
    json.dump(meta, open(f"{OUT_DIR}/metadata.json", "w"), indent=2)

TAG = "classic_v2"
if RUN_FULL:
    assert test is not None, "attach the competition data and rerun Cell 1"
    final = ClassicModel(CONFIGS[BEST]).fit(train)
    parts = [final.predict(test.iloc[i:i + 100000]) for i in range(0, len(test), 100000)]
    tst = {k: np.vstack([p[k] for p in parts]) for k in parts[0]}
    save_artifacts(BEST, TAG, oofs[BEST], tst, test["id"].values)
    lab, act, techs = decode(tst["label"], tst["action"], tst["tech"], True)
    mp = tst["label"].max(1, keepdims=True); ok = (oofs[BEST]["label"].argmax(1) == train["y"].values).astype(int)
    cal = LogisticRegression(C=10, random_state=SEED).fit(oofs[BEST]["label"].max(1, keepdims=True), ok)
    conf = np.clip(cal.predict_proba(mp)[:, 1], 0.05, 0.98)
    sub = pd.DataFrame({"id": test["id"], "label": lab.astype(int), "techniques": techs, "action": act.astype(int), "confidence": np.round(conf, 4)})
    sub.to_csv(f"{OUT_DIR}/submission_{TAG}.csv", index=False)
    print("saved submission_%s.csv" % TAG, sub.shape, "| label mix:", np.round(np.bincount(lab, minlength=6) / len(lab), 3))
else:
    save_artifacts(BEST, TAG, oofs[BEST])
    print("validation-only: saved oof artifacts for", BEST, "(set RUN_FULL = True for full test inference)")


rows 86 | groups 85
E0 lr baseline             done    24s
E1 lr +noise x1            done    49s
E2 lr +noise x2            done    87s
E3 svc +noise x1           done    96s
E4 lr +struct +noise x1    done   121s
E5 lr+svc +noise x1        done   157s
E6 lr+svc+cnb +noise x1    done   190s

Mean over 3 repeated grouped 5-fold runs (real rows only; proxies, not Kaggle scores):
                          A_F1  B_F1(free)  B_F1(NONE for 0,5)  C_F1(free)  C_F1(0 for 0,5)  A_F1_noisy  calib(1-Brier)
E0 lr baseline           0.558       0.404               0.396       0.440            0.410       0.493           0.779
E1 lr +noise x1          0.572       0.416               0.417       0.444            0.406       0.513           0.774
E2 lr +noise x2          0.574       0.435               0.424       0.448            0.407       0.514           0.772
E3 svc +noise x1         0.574       0.386               0.373       0.441            0.413       0.502           0.771
E4 lr +struct +nois